In [1]:

# Phase 4 Features & Splits: Leakage Audit
import pandas as pd
import yaml

print("Task: Phase 4 Leakage Audit started")

df_cold = pd.read_parquet('../data/processed/features_cold.parquet')
df_full = pd.read_parquet('../data/processed/features_full.parquet')

# Features in cold (excluding identifiers and target)
exclude = ['post_id', 'username', 'posted_at', 'split', 'time_split', 'y', 'engagement_rate', 'original_category']
cold_features = [c for c in df_cold.columns if c not in exclude]
full_features = [c for c in df_full.columns if c not in exclude]

print("CHECKPOINT 1 - Feature lists extracted: DONE")

# Leakage Audit Table
audit = []
for f in full_features:
    known = "Yes"
    if f in ['engagement_rate', 'y', 'likes', 'comments']:
        known = "No"
    
    audit.append({'Feature': f, 'Known before publishing?': known})

audit_df = pd.DataFrame(audit)
print("\nLeakage Audit:")
print(audit_df.to_string(index=False))

# Assert no leakage
assert not any(audit_df['Known before publishing?'] == 'No'), "LEAKAGE DETECTED! Found feature not known before publishing."
print("\nCHECKPOINT 2 - Leakage audit passed: DONE")

# Write allowlist to config.yaml
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

config['feature_allowlist'] = {
    'cold': cold_features,
    'full': full_features
}

with open('../config.yaml', 'w') as f:
    yaml.dump(config, f, sort_keys=False)

print("CHECKPOINT 3 - config.yaml updated with feature_allowlist: DONE")
print("DONE!! - Phase 4 Leakage audit completed successfully.")


Task: Phase 4 Leakage Audit started


CHECKPOINT 1 - Feature lists extracted: DONE

Leakage Audit:
                  Feature Known before publishing?
            log_followers                      Yes
              hour_of_day                      Yes
              day_of_week                      Yes
               is_weekend                      Yes
            hashtag_count                      Yes
           caption_length                      Yes
            usertag_count                      Yes
                  n_media                      Yes
          category_beauty                      Yes
          category_family                      Yes
         category_fashion                      Yes
         category_fitness                      Yes
            category_food                      Yes
        category_interior                      Yes
           category_other                      Yes
             category_pet                      Yes
          category_travel                      Yes
      media_type_caro